# Task 5: NLP & Narrative Analysis

**Project:** Coordinated Amplification and Misinformation Detection in Global YouTube Conflict Narratives  
**Course:** CS 418 (UIC)

This notebook performs NLP analysis on YouTube comments related to the Russia-Ukraine conflict:
1. Language detection across the multilingual comment corpus
2. Multilingual sentiment analysis using XLM-RoBERTa
3. Topic modeling with BERTopic to discover narrative themes
4. Zero-shot misinformation classification
5. Integration with network/anomaly detection results from Tasks 2-3

**Hypothesis H3:** Narratives evolve in alignment with key conflict events.

## 1. Setup & Data Loading

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from tqdm import tqdm
import torch
import warnings
warnings.filterwarnings('ignore')

from utils.bq_helpers import load_comments, load_videos, COLORS
from utils.conflict_timeline import EVENTS_DF, add_event_annotations

# Check device
if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'
print(f'Using device: {DEVICE}')
print('Libraries loaded successfully')

Using device: mps
Libraries loaded successfully


In [2]:
# Load stratified sample of comments (~200K)
print('Loading comment sample from BigQuery...')
comments = load_comments(sample_frac=0.2)
print(f'Comments loaded: {comments.shape}')
print(f'Date range: {comments["published_at"].min()} to {comments["published_at"].max()}')
print(f'\nNull counts:')
print(comments.isnull().sum())

# Drop null comment text
comments = comments.dropna(subset=['comment_text']).copy()
# Truncate long comments for model input
comments['text_clean'] = comments['comment_text'].astype(str).str[:512]
print(f'\nComments after cleanup: {len(comments):,}')

Loading comment sample from BigQuery...


Comments loaded: (33796, 7)
Date range: 2018-04-21 00:25:19+00:00 to 2026-03-14 21:48:45+00:00

Null counts:
video_id              0
comment_id            0
comment_text         18
author_name           0
author_channel_id     0
published_at          0
like_count            0
dtype: int64

Comments after cleanup: 33,778


## 2. Language Detection

In [3]:
from langdetect import detect, LangDetectException

def safe_detect(text):
    try:
        lang = detect(str(text)[:200])
        if lang in ('en',):
            return 'EN'
        elif lang in ('ru',):
            return 'RU'
        elif lang in ('uk',):
            return 'UK'
        else:
            return 'OTHER'
    except (LangDetectException, Exception):
        return 'OTHER'

# Detect on sample (full detection is slow, so use first 50K)
detect_sample = comments.head(50000).copy()
print(f'Detecting language for {len(detect_sample):,} comments...')
tqdm.pandas(desc='Language detection')
detect_sample['detected_lang'] = detect_sample['text_clean'].progress_apply(safe_detect)

# Apply detected language proportions to full dataset via sampling
lang_dist = detect_sample['detected_lang'].value_counts(normalize=True)
print(f'\nLanguage distribution:')
print(lang_dist)

# Assign to full dataset
comments['detected_lang'] = 'OTHER'  # default
comments.loc[detect_sample.index, 'detected_lang'] = detect_sample['detected_lang']

Detecting language for 33,778 comments...


Language detection:   0%|          | 0/33778 [00:00<?, ?it/s]

Language detection:   0%|          | 4/33778 [00:00<14:20, 39.24it/s]

Language detection:   0%|          | 75/33778 [00:00<01:18, 429.63it/s]

Language detection:   0%|          | 136/33778 [00:00<01:10, 477.64it/s]

...

Language detection:  99%|█████████▉| 33577/33778 [00:41<00:00, 837.72it/s]

Language detection: 100%|█████████▉| 33661/33778 [00:41<00:00, 814.97it/s]

Language detection: 100%|█████████▉| 33743/33778 [00:41<00:00, 770.38it/s]

Language detection: 100%|██████████| 33778/33778 [00:41<00:00, 809.53it/s]


Language distribution:
detected_lang
OTHER    0.536621
EN       0.462313
RU       0.001066
Name: proportion, dtype: float64


In [4]:
# Language distribution bar chart
lang_counts = detect_sample['detected_lang'].value_counts().reset_index()
lang_counts.columns = ['Language', 'Count']
fig = px.bar(lang_counts, x='Language', y='Count',
    title='Language Distribution in Comment Sample',
    color='Language', color_discrete_map={'EN': '#1f77b4', 'RU': '#d62728', 'UK': '#ff7f0e', 'OTHER': '#7f7f7f'})
fig.update_layout(height=400, width=700)
fig.show()

## 3. Sentiment Analysis

Using `cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual` for multilingual sentiment classification.

In [5]:
from transformers import pipeline

# Load sentiment model
print('Loading sentiment model...')
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual',
    device=DEVICE,
    top_k=1,
    truncation=True,
    max_length=512
)
print('Sentiment model loaded')

Loading sentiment model...


config.json:   0%|          | 0.00/982 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Device set to use mps


Sentiment model loaded


In [6]:
# Run sentiment analysis in batches
# Use a manageable sample size for inference
sent_sample = comments.head(50000).copy()
texts = sent_sample['text_clean'].tolist()

batch_size = 64
sentiments = []

print(f'Running sentiment analysis on {len(texts):,} comments...')
for i in tqdm(range(0, len(texts), batch_size), desc='Sentiment'):
    batch = texts[i:i+batch_size]
    try:
        results = sentiment_pipe(batch)
        for r in results:
            sentiments.append(r[0]['label'])
    except Exception as e:
        sentiments.extend(['neutral'] * len(batch))

sent_sample['sentiment'] = sentiments[:len(sent_sample)]
print(f'\nSentiment distribution:')
print(sent_sample['sentiment'].value_counts())

Running sentiment analysis on 33,778 comments...


Sentiment:   0%|          | 0/528 [00:00<?, ?it/s]

Sentiment:   0%|          | 1/528 [00:08<1:18:03,  8.89s/it]

Sentiment:   0%|          | 2/528 [00:11<43:37,  4.98s/it]  

Sentiment:   1%|          | 3/528 [00:12<30:05,  3.44s/it]

...

Sentiment: 100%|█████████▉| 526/528 [12:28<00:03,  1.83s/it]

Sentiment: 100%|█████████▉| 527/528 [12:30<00:01,  1.84s/it]

Sentiment: 100%|██████████| 528/528 [12:31<00:00,  1.72s/it]

Sentiment: 100%|██████████| 528/528 [12:31<00:00,  1.42s/it]


Sentiment distribution:
sentiment
positive    15914
negative    11198
neutral      6666
Name: count, dtype: int64


In [7]:
# Sentiment distribution by language
lang_sent = sent_sample[sent_sample['detected_lang'] != 'OTHER']
if len(lang_sent) > 0:
    cross = pd.crosstab(lang_sent['detected_lang'], lang_sent['sentiment'], normalize='index')
    fig = px.bar(cross.reset_index().melt(id_vars='detected_lang'),
        x='detected_lang', y='value', color='sentiment',
        title='Sentiment Distribution by Language',
        labels={'value': 'Proportion', 'detected_lang': 'Language'},
        color_discrete_map={'positive': '#2ca02c', 'negative': '#d62728', 'neutral': '#7f7f7f'})
    fig.update_layout(height=450, width=800)
    fig.show()

In [8]:
# Weekly sentiment time series
sent_sample['week'] = sent_sample['published_at'].dt.to_period('W').dt.start_time.dt.tz_localize('UTC')
weekly_sent = sent_sample.groupby('week')['sentiment'].value_counts(normalize=True).unstack(fill_value=0).reset_index()

fig = go.Figure()
for col, color in [('positive', '#2ca02c'), ('negative', '#d62728'), ('neutral', '#7f7f7f')]:
    if col in weekly_sent.columns:
        fig.add_trace(go.Scatter(x=weekly_sent['week'], y=weekly_sent[col],
            mode='lines', name=col.title(), line=dict(color=color)))

fig = add_event_annotations(fig)
fig.update_layout(title='Weekly Sentiment Proportions with Conflict Events',
    height=500, width=1100, xaxis_title='Date', yaxis_title='Proportion')
fig.show()

## 4. Topic Modeling with BERTopic

In [9]:
try:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    
    # Use English comments for cleaner topics
    en_comments = sent_sample[sent_sample['detected_lang'] == 'EN'].copy()
    topic_texts = en_comments['text_clean'].tolist()
    
    # Limit for performance
    max_docs = min(50000, len(topic_texts))
    topic_texts = topic_texts[:max_docs]
    topic_dates = en_comments['published_at'].head(max_docs).tolist()
    
    print(f'Fitting BERTopic on {len(topic_texts):,} English comments...')
    
    # Initialize BERTopic
    embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    topic_model = BERTopic(
        embedding_model=embedding_model,
        nr_topics=20,
        verbose=True,
        min_topic_size=50,
    )
    
    topics, probs = topic_model.fit_transform(topic_texts)
    
    print(f'\nTopics found: {len(set(topics)) - 1}')  # -1 for outlier topic
    topic_info = topic_model.get_topic_info()
    print(topic_info.head(20))
    
    BERTOPIC_AVAILABLE = True

except ImportError:
    print('BERTopic not installed. Install with: pip install bertopic sentence-transformers')
    print('Skipping topic modeling.')
    BERTOPIC_AVAILABLE = False

Fitting BERTopic on 15,616 English comments...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

2026-03-14 17:25:58,985 - BERTopic - Topic reduction - Reducing number of topics


2026-03-14 17:25:58,996 - BERTopic - Representation - Fine-tuning topics using representation models.


2026-03-14 17:25:59,098 - BERTopic - Representation - Completed ✓


2026-03-14 17:25:59,099 - BERTopic - Topic reduction - Reduced number of topics from 35 to 20



Topics found: 19
    Topic  Count                             Name  \
0      -1   5930                 -1_the_to_and_of   
1       0   2132              0_you_he_thank_this   
2       1   1262                  1_to_it_the_for   
3       2    757            2_iran_russia_war_the   
4       3    750          3_video_videos_you_your   
5       4    630                  4_the_you_to_it   
6       5    536                  5_and_the_to_dr   
7       6    534                6_trump_the_is_he   
8       7    509           7_the_of_universe_that   
9       8    505    8_india_pakistan_indian_china   
10      9    474                   9_my_to_me_and   
11     10    418      10_animals_animal_vegan_and   
12     11    312             11_allah_and_god_the   
13     12    237         12_music_violin_song_the   
14     13    195          13_we_players_united_he   
15     14    127     14_news_media_reporting_good   
16     15    112       15_analysis_very_sir_great   
17     16     84  16_delicio

In [10]:
if BERTOPIC_AVAILABLE:
    # Topic sizes bar chart
    topic_info_plot = topic_info[topic_info['Topic'] != -1].head(20)
    fig = px.bar(topic_info_plot, x='Count', y='Name', orientation='h',
        title='Top 20 Topics by Size',
        labels={'Count': 'Number of Comments', 'Name': 'Topic'})
    fig.update_layout(height=600, width=1000, yaxis=dict(autorange='reversed'))
    fig.show()

In [11]:
if BERTOPIC_AVAILABLE:
    # Topics over time — tests H3 directly
    print('Computing topics over time...')
    topics_over_time = topic_model.topics_over_time(
        topic_texts, topic_dates, nr_bins=30
    )
    
    # Plot top topics over time
    top_topic_ids = topic_info[topic_info['Topic'] != -1].head(10)['Topic'].tolist()
    tot_filtered = topics_over_time[topics_over_time['Topic'].isin(top_topic_ids)]
    
    fig = px.line(tot_filtered, x='Timestamp', y='Frequency',
        color='Name' if 'Name' in tot_filtered.columns else 'Topic',
        title='Narrative Evolution: Top Topics Over Time (H3)')
    fig = add_event_annotations(fig)
    fig.update_layout(height=600, width=1100, xaxis_title='Date', yaxis_title='Frequency')
    fig.show()
    
    print('Topics over time computed.')

Computing topics over time...


0it [00:00, ?it/s]

8it [00:00, 67.98it/s]

15it [00:00, 53.49it/s]

21it [00:00, 47.60it/s]

26it [00:00, 38.33it/s]

29it [00:00, 35.88it/s]

Topics over time computed.


## 5. Misinformation Classification

Zero-shot classification with `facebook/bart-large-mnli`.

In [12]:
# Load zero-shot classifier
print('Loading zero-shot classification model...')
misinfo_pipe = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=DEVICE
)

candidate_labels = ['misinformation', 'propaganda', 'factual reporting', 'opinion']

# Classify a sample (zero-shot is slow, so use smaller sample)
misinfo_sample_size = min(5000, len(sent_sample))
misinfo_texts = sent_sample['text_clean'].head(misinfo_sample_size).tolist()

print(f'Classifying {misinfo_sample_size:,} comments for misinformation...')
misinfo_labels = []
misinfo_scores = []

batch_size = 8  # smaller batch for zero-shot
for i in tqdm(range(0, len(misinfo_texts), batch_size), desc='Misinfo classification'):
    batch = misinfo_texts[i:i+batch_size]
    try:
        results = misinfo_pipe(batch, candidate_labels=candidate_labels)
        if not isinstance(results, list):
            results = [results]
        for r in results:
            misinfo_labels.append(r['labels'][0])
            misinfo_scores.append(r['scores'][0])
    except Exception:
        misinfo_labels.extend(['opinion'] * len(batch))
        misinfo_scores.extend([0.0] * len(batch))

sent_sample.loc[sent_sample.index[:misinfo_sample_size], 'misinfo_label'] = misinfo_labels[:misinfo_sample_size]
sent_sample.loc[sent_sample.index[:misinfo_sample_size], 'misinfo_score'] = misinfo_scores[:misinfo_sample_size]

print(f'\nMisinformation classification distribution:')
print(sent_sample['misinfo_label'].value_counts())

Loading zero-shot classification model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Misinfo classification: 100%|█████████▉| 623/625 [11:33<00:02,  1.03s/it]

Misinfo classification: 100%|█████████▉| 624/625 [11:34<00:01,  1.01s/it]

Misinfo classification: 100%|██████████| 625/625 [11:35<00:00,  1.01it/s]

Misinfo classification: 100%|██████████| 625/625 [11:35<00:00,  1.11s/it]


Misinformation classification distribution:
misinfo_label
nan                  28778
opinion               2951
misinformation        1161
factual reporting      801
propaganda              87
Name: count, dtype: int64


In [13]:
# Misinformation distribution pie chart
misinfo_dist = sent_sample['misinfo_label'].value_counts().reset_index()
misinfo_dist.columns = ['Label', 'Count']
fig = px.pie(misinfo_dist, values='Count', names='Label',
    title='Comment Classification Distribution',
    color_discrete_map={'misinformation': '#d62728', 'propaganda': '#ff7f0e',
                        'factual reporting': '#2ca02c', 'opinion': '#1f77b4'})
fig.update_layout(height=450, width=600)
fig.show()

In [14]:
# Misinformation proportion over time
misinfo_subset = sent_sample.dropna(subset=['misinfo_label']).copy()
misinfo_subset['is_misinfo'] = misinfo_subset['misinfo_label'].isin(['misinformation', 'propaganda']).astype(int)
misinfo_subset['week'] = misinfo_subset['published_at'].dt.to_period('W').dt.start_time.dt.tz_localize('UTC')

weekly_misinfo = misinfo_subset.groupby('week').agg(
    misinfo_rate=('is_misinfo', 'mean'),
    total=('is_misinfo', 'count')
).reset_index()
weekly_misinfo = weekly_misinfo[weekly_misinfo['total'] >= 10]  # min sample size

fig = go.Figure()
fig.add_trace(go.Scatter(x=weekly_misinfo['week'], y=weekly_misinfo['misinfo_rate'],
    mode='lines+markers', name='Misinfo Rate',
    line=dict(color=COLORS['anomaly']), marker=dict(size=4)))
fig = add_event_annotations(fig)
fig.update_layout(title='Weekly Misinformation/Propaganda Rate with Conflict Events',
    height=500, width=1100, xaxis_title='Date', yaxis_title='Misinfo + Propaganda Rate')
fig.show()

## 6. Integration with Other Tasks

In [15]:
# Try loading outputs from Tasks 2 and 3
try:
    channel_clusters = pd.read_csv('../task2_network_analysis/outputs/channel_clusters.csv')
    print(f'Loaded channel clusters: {channel_clusters.shape}')
    has_clusters = True
except FileNotFoundError:
    print('Channel clusters not found. Run Task 2 first.')
    has_clusters = False

try:
    anomaly_scores = pd.read_csv('../task3_engagement_analysis_and_anomaly_detection/outputs/video_anomaly_scores.csv')
    print(f'Loaded anomaly scores: {anomaly_scores.shape}')
    has_anomalies = True
except FileNotFoundError:
    print('Anomaly scores not found. Run Task 3 first.')
    has_anomalies = False

Loaded channel clusters: (263, 7)
Loaded anomaly scores: (84307, 10)


In [16]:
# Cross-reference: sentiment by channel cluster
if has_clusters:
    # Load video data to get video_id -> channel_id mapping
    video_channels = load_videos(['video_id', 'channel_id'])
    
    sent_with_channel = sent_sample.merge(video_channels, on='video_id', how='left')
    sent_with_channel = sent_with_channel.merge(
        channel_clusters[['channel_id', 'cluster_id']], on='channel_id', how='left')
    
    # Sentiment by cluster
    cluster_sent = sent_with_channel.dropna(subset=['cluster_id', 'sentiment'])
    if len(cluster_sent) > 0:
        top_clusters = cluster_sent['cluster_id'].value_counts().head(8).index
        cross = pd.crosstab(
            cluster_sent[cluster_sent['cluster_id'].isin(top_clusters)]['cluster_id'].astype(int),
            cluster_sent[cluster_sent['cluster_id'].isin(top_clusters)]['sentiment'],
            normalize='index'
        )
        fig = px.bar(cross.reset_index().melt(id_vars='cluster_id'),
            x='cluster_id', y='value', color='sentiment',
            title='Sentiment Distribution by Channel Cluster',
            labels={'value': 'Proportion', 'cluster_id': 'Cluster'},
            color_discrete_map={'positive': '#2ca02c', 'negative': '#d62728', 'neutral': '#7f7f7f'})
        fig.update_layout(height=450, width=900)
        fig.show()
else:
    print('Skipping cluster integration (no cluster data).')

In [17]:
# Cross-reference: sentiment of comments on anomalous videos
if has_anomalies:
    anomalous_videos = set(anomaly_scores[anomaly_scores['anomaly_label'] == 1]['video_id'])
    sent_sample['on_anomalous_video'] = sent_sample['video_id'].isin(anomalous_videos)
    
    anom_sent = sent_sample.groupby('on_anomalous_video')['sentiment'].value_counts(normalize=True).unstack(fill_value=0)
    print('Sentiment on anomalous vs normal videos:')
    print(anom_sent)
    
    fig = px.bar(anom_sent.reset_index().melt(id_vars='on_anomalous_video'),
        x='on_anomalous_video', y='value', color='sentiment',
        title='Sentiment: Anomalous vs Normal Videos',
        labels={'value': 'Proportion', 'on_anomalous_video': 'Anomalous Video'},
        color_discrete_map={'positive': '#2ca02c', 'negative': '#d62728', 'neutral': '#7f7f7f'})
    fig.update_layout(height=400, width=700)
    fig.show()
else:
    print('Skipping anomaly integration (no anomaly data).')

Sentiment on anomalous vs normal videos:
sentiment           negative   neutral  positive
on_anomalous_video                              
False               0.349781  0.203472  0.446747
True                0.253662  0.171237  0.575101


## 7. H3 Conclusion

### Do narratives shift predictably around conflict events?

Based on our analysis:

1. **Sentiment Trends**: The weekly sentiment time series shows how the emotional tone of comments shifts around major conflict events. Look for sentiment dips (more negative) around events like the Bucha massacre and sentiment recovery periods.

2. **Topic Evolution**: BERTopic's `topics_over_time()` reveals which narrative themes emerge and fade around key events. New topics may emerge after major events (e.g., war crimes narratives after Bucha, mobilization narratives after Putin's September 2022 announcement).

3. **Misinformation Patterns**: The weekly misinformation rate overlaid with conflict events shows whether propaganda and misinformation spike during high-attention periods.

4. **Cross-Task Integration**: Comparing sentiment across channel clusters (Task 2) and on anomalous videos (Task 3) reveals whether coordinated networks drive different narrative patterns.

**Verdict:** Examine the topic evolution and sentiment trend charts above to assess whether narrative shifts align with conflict events, supporting or refuting H3.

## 8. Save Outputs

In [18]:
import os
os.makedirs('outputs', exist_ok=True)

# Save comment classifications
output_cols = ['comment_id', 'video_id', 'detected_lang', 'sentiment']
if 'misinfo_label' in sent_sample.columns:
    output_cols.extend(['misinfo_label', 'misinfo_score'])
if BERTOPIC_AVAILABLE:
    # Add topic IDs
    sent_sample_out = sent_sample.head(len(topics)).copy()
    sent_sample_out['topic_id'] = topics[:len(sent_sample_out)]
    output_cols.append('topic_id')
else:
    sent_sample_out = sent_sample.copy()

valid_cols = [c for c in output_cols if c in sent_sample_out.columns]
sent_sample_out[valid_cols].to_csv('outputs/comment_classifications.csv', index=False)
print(f'Saved outputs/comment_classifications.csv ({len(sent_sample_out):,} rows)')

# Save topics over time
if BERTOPIC_AVAILABLE:
    topics_over_time.to_csv('outputs/topics_over_time.csv', index=False)
    print(f'Saved outputs/topics_over_time.csv ({len(topics_over_time):,} rows)')

# Save sentiment time series
weekly_sent_out = weekly_sent.copy()
weekly_sent_out.columns.name = None
weekly_sent_out = weekly_sent_out.rename(columns={'week': 'date'})
weekly_sent_out.to_csv('outputs/sentiment_timeseries.csv', index=False)
print(f'Saved outputs/sentiment_timeseries.csv ({len(weekly_sent_out):,} rows)')

print('\nDone! All outputs saved.')

Saved outputs/comment_classifications.csv (15,616 rows)
Saved outputs/topics_over_time.csv (297 rows)
Saved outputs/sentiment_timeseries.csv (352 rows)

Done! All outputs saved.
